In [152]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

Connected to gold.duckdb


In [ ]:
# Data freshness -- last IPL match ingested into gold.duckdb
q("""
SELECT
    m.match_id,
    m.match_date,
    m.season,
    m.team_1,
    m.team_2,
    coalesce(m.event_stage, 'League')        AS stage,
    coalesce(m.outcome_winner, 'No result')  AS result
FROM main_marts.fct_match_results m
WHERE m.event_name ILIKE '%indian premier league%'
ORDER BY m.match_date DESC
LIMIT 1
""")

In [153]:
# ── Change player name here to analyse anyone ────────────────────────────────
PLAYER = "SK Raina"
# ────────────────────────────────────────────────────────────────────────────
print(f"Showing IPL stats for: {PLAYER}")

Showing IPL stats for: SK Raina


## Batting

In [154]:
# Career IPL batting summary
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number FROM bat
    UNION
    SELECT match_id, innings_number FROM dis
    UNION
    -- Player reached the crease as non-striker but innings ended before they faced a ball.
    -- Only included when the player does not bat anywhere else in the same match,
    -- which prevents Cricsheet data errors (wrong non_striker in opposing team's innings).
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
""")

,mat,inns,no,runs,hs,ave,bf,sr,100,50,0,4s,6s
0,200,200,30.0,5528.0,100.0,32.52,4043.0,136.73,1.0,39.0,8.0,506.0,203.0


In [155]:
# IPL batting by season
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season, d.batting_team,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season, d.batting_team
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season, batting_team FROM bat
    UNION
    SELECT match_id, innings_number, season, batting_team FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season, a.batting_team,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    mode(batting_team)                                          AS team,
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
GROUP BY season
ORDER BY season
""")

,season,team,mat,inns,no,runs,hs,ave,bf,sr,100,50,0,4s,6s
0,2007/08,Chennai Super Kings,14,14,3.0,421.0,55.0,38.27,295.0,142.71,0.0,3.0,0.0,35.0,18.0
1,2009,Chennai Super Kings,14,14,0.0,434.0,98.0,31.00,308.0,140.91,0.0,2.0,0.0,37.0,21.0
2,2009/10,Chennai Super Kings,16,16,5.0,520.0,83.0,47.27,364.0,142.86,0.0,4.0,0.0,45.0,22.0
3,2011,Chennai Super Kings,16,16,2.0,438.0,73.0,31.29,325.0,134.77,0.0,4.0,1.0,36.0,17.0
4,2012,Chennai Super Kings,18,18,1.0,441.0,73.0,25.94,325.0,135.69,0.0,1.0,2.0,36.0,19.0
5,2013,Chennai Super Kings,17,17,4.0,548.0,100.0,42.15,365.0,150.14,1.0,4.0,3.0,50.0,18.0
6,2014,Chennai Super Kings,16,16,3.0,523.0,87.0,40.23,359.0,145.68,0.0,5.0,0.0,51.0,19.0
7,2015,Chennai Super Kings,17,17,2.0,374.0,62.0,24.93,305.0,122.62,0.0,2.0,1.0,31.0,16.0
8,2016,Gujarat Lions,15,15,1.0,399.0,75.0,28.50,312.0,127.88,0.0,3.0,0.0,39.0,10.0
9,2017,Gujarat Lions,14,14,3.0,442.0,84.0,40.18,307.0,143.97,0.0,3.0,0.0,42.0,13.0


In [156]:
# Top 20 highest IPL innings
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
)
SELECT
    b.season, b.match_date, b.batting_team, b.bowling_team AS vs,
    b.runs,
    (d.match_id IS NULL) AS not_out,
    b.balls,
    round(b.runs * 100.0 / nullif(b.balls, 0), 1) AS sr,
    b.fours, b.sixes
FROM bat b
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY b.runs DESC
LIMIT 20
""")

,season,match_date,batting_team,vs,runs,not_out,balls,sr,fours,sixes
0,2013,2013-05-02,Chennai Super Kings,Punjab Kings,100.0,True,53.0,188.7,7.0,6.0
1,2013,2013-05-08,Chennai Super Kings,Sunrisers Hyderabad,99.0,True,52.0,190.4,11.0,3.0
2,2009,2009-04-30,Chennai Super Kings,Rajasthan Royals,98.0,False,55.0,178.2,10.0,5.0
3,2014,2014-05-30,Chennai Super Kings,Punjab Kings,87.0,False,25.0,348.0,12.0,6.0
4,2017,2017-04-21,Gujarat Lions,Kolkata Knight Riders,84.0,False,46.0,182.6,9.0,4.0
5,2009/10,2010-03-25,Chennai Super Kings,Mumbai Indians,83.0,True,52.0,159.6,7.0,3.0
6,2013,2013-05-21,Chennai Super Kings,Mumbai Indians,82.0,True,42.0,195.2,5.0,5.0
7,2009/10,2010-04-13,Chennai Super Kings,Kolkata Knight Riders,78.0,True,39.0,200.0,11.0,3.0
8,2017,2017-05-04,Gujarat Lions,Delhi Capitals,77.0,False,43.0,179.1,5.0,4.0
9,2016,2016-04-21,Gujarat Lions,Sunrisers Hyderabad,75.0,False,51.0,147.1,9.0,0.0


## Bowling

In [157]:
# Career IPL bowling summary
q(f"""
WITH bowl AS (
    SELECT
        d.match_id,
        d.innings_number,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts,
        sum(CASE WHEN d.extras_wides > 0 THEN 1 ELSE 0 END)                            AS wides,
        sum(CASE WHEN d.extras_noballs > 0 THEN 1 ELSE 0 END)                          AS no_balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
maidens AS (
    SELECT d.match_id, d.innings_number, d.over_number,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes) AS over_runs
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.over_number
    HAVING sum(d.runs_total - d.extras_byes - d.extras_legbyes) = 0
       AND count(*) >= 6
)
SELECT
    count(DISTINCT b.match_id)                                                          AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(b.legal_del) / 6)::integer || '.' || (sum(b.legal_del) % 6))::varchar   AS overs,
    sum(b.legal_del)                                                                    AS balls,
    (SELECT count(*) FROM maidens)                                                      AS mdns,
    sum(b.runs_conceded)                                                                AS runs,
    sum(b.wkts)                                                                         AS wkts,
    max_by(b.wkts || '/' || b.runs_conceded, b.wkts * 1000 - b.runs_conceded)          AS bbi,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wkts), 0), 2)                    AS ave,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_del), 0), 2)                 AS econ,
    round(sum(b.legal_del)::double / nullif(sum(b.wkts), 0), 2)                        AS sr,
    sum(CASE WHEN b.wkts >= 4 THEN 1 ELSE 0 END)                                       AS "4w",
    sum(CASE WHEN b.wkts >= 5 THEN 1 ELSE 0 END)                                       AS "5w"
FROM bowl b
""")

,mat,inns,overs,balls,mdns,runs,wkts,bbi,ave,econ,sr,4w,5w
0,69,69,151.2,908.0,0,1118.0,25.0,2/0,44.72,7.39,36.32,0.0,0.0


In [158]:
# IPL bowling by season
q(f"""
WITH bowl AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
)
SELECT
    season,
    count(DISTINCT match_id)                                                            AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(legal_del) / 6)::integer || '.' || (sum(legal_del) % 6))::varchar       AS overs,
    sum(legal_del)                                                                      AS balls,
    sum(runs_conceded)                                                                  AS runs,
    sum(wkts)                                                                           AS wkts,
    max_by(wkts || '/' || runs_conceded, wkts * 1000 - runs_conceded)                  AS bbi,
    round(sum(runs_conceded)::double / nullif(sum(wkts), 0), 2)                         AS ave,
    round(sum(runs_conceded) * 6.0 / nullif(sum(legal_del), 0), 2)                     AS econ,
    round(sum(legal_del)::double / nullif(sum(wkts), 0), 2)                             AS sr,
    sum(CASE WHEN wkts >= 4 THEN 1 ELSE 0 END)                                          AS "4w",
    sum(CASE WHEN wkts >= 5 THEN 1 ELSE 0 END)                                          AS "5w"
FROM bowl
GROUP BY season
ORDER BY season
""")

,season,mat,inns,overs,balls,runs,wkts,bbi,ave,econ,sr,4w,5w
0,2007/08,2,2,3.2,20.0,30.0,1.0,1/20,30.00,9.00,20.00,0.0,0.0
1,2009,10,10,27.4,166.0,164.0,7.0,2/17,23.43,5.93,23.71,0.0,0.0
2,2009/10,12,12,23.5,143.0,178.0,6.0,1/0,29.67,7.47,23.83,0.0,0.0
3,2011,12,12,27.3,165.0,201.0,4.0,2/0,50.25,7.31,41.25,0.0,0.0
4,2012,7,7,14.0,84.0,105.0,2.0,1/5,52.50,7.50,42.00,0.0,0.0
5,2013,2,2,2.0,12.0,23.0,1.0,1/4,23.00,11.50,12.00,0.0,0.0
6,2014,8,8,18.0,108.0,134.0,1.0,1/17,134.00,7.44,108.00,0.0,0.0
7,2015,5,5,12.0,72.0,93.0,2.0,1/29,46.50,7.75,36.00,0.0,0.0
8,2016,4,4,10.0,60.0,82.0,0.0,0/15,NaN,8.20,NaN,0.0,0.0
9,2017,6,6,12.0,72.0,102.0,1.0,1/11,102.00,8.50,72.00,0.0,0.0


In [159]:
# Best bowling innings in IPL (top 20)
q(f"""
SELECT
    m.season,
    d.match_date,
    d.bowling_team,
    CASE WHEN m.team_1 = d.bowling_team THEN m.team_2 ELSE m.team_1 END AS vs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                              AS wkts,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                 AS runs,
    sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
             THEN 1 ELSE 0 END)                                          AS balls,
    (floor(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                    THEN 1 ELSE 0 END) / 6)::integer
     || '.' ||
     (sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
               THEN 1 ELSE 0 END) % 6))::varchar                         AS overs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
GROUP BY m.season, d.match_id, d.innings_number, d.match_date, d.bowling_team, m.team_1, m.team_2
ORDER BY wkts DESC, runs
LIMIT 20
""")

,season,match_date,bowling_team,vs,wkts,runs,balls,overs
0,2011,2011-05-09,Chennai Super Kings,Rajasthan Royals,2.0,0.0,3.0,0.3
1,2009,2009-05-20,Chennai Super Kings,Punjab Kings,2.0,17.0,24.0,4.0
2,2009,2009-04-27,Chennai Super Kings,Sunrisers Hyderabad,2.0,18.0,24.0,4.0
3,2009/10,2010-04-22,Chennai Super Kings,Sunrisers Hyderabad,1.0,0.0,2.0,0.2
4,2011,2011-04-08,Chennai Super Kings,Kolkata Knight Riders,1.0,3.0,6.0,1.0
5,2013,2013-05-08,Chennai Super Kings,Sunrisers Hyderabad,1.0,4.0,6.0,1.0
6,2012,2012-05-04,Chennai Super Kings,Sunrisers Hyderabad,1.0,5.0,6.0,1.0
7,2012,2012-04-07,Chennai Super Kings,Sunrisers Hyderabad,1.0,10.0,12.0,2.0
8,2017,2017-04-21,Gujarat Lions,Kolkata Knight Riders,1.0,11.0,12.0,2.0
9,2009,2009-04-30,Chennai Super Kings,Rajasthan Royals,1.0,11.0,12.0,2.0


In [160]:
# Match-by-match bowling figures — 2025 IPL
q(f"""
SELECT
    d.match_date,
    CASE WHEN m.team_1 = d.bowling_team THEN m.team_2 ELSE m.team_1 END       AS vs,
    (floor(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                    THEN 1 ELSE 0 END) / 6)::integer
     || '.' ||
     (sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
               THEN 1 ELSE 0 END) % 6))::varchar                               AS overs,
    sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
             THEN 1 ELSE 0 END)                                                AS balls,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                       AS runs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                                    AS wkts,
    sum(CASE WHEN d.extras_wides  > 0 THEN 1 ELSE 0 END)                      AS wides,
    sum(CASE WHEN d.extras_noballs > 0 THEN 1 ELSE 0 END)                     AS nb,
    round(sum(d.runs_total - d.extras_byes - d.extras_legbyes) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                            THEN 1 ELSE 0 END), 0), 2)                         AS econ,
    d.match_id
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '2025'
  AND d.innings_number IN (1, 2)
GROUP BY d.match_id, d.match_date, d.innings_number, d.bowling_team, m.team_1, m.team_2
ORDER BY d.match_date
""")

,match_date,vs,overs,balls,runs,wkts,wides,nb,econ,match_id


In [161]:
# Diagnostic: raw bowling innings for 2025 — shows all 11 rows and exposes any NULL bowling_team / match_date
q(f"""
SELECT
    d.match_id,
    d.innings_number,
    d.match_date,
    d.bowling_team,
    m.team_1,
    m.team_2,
    count(*)                                                                            AS deliveries,
    sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)       AS legal_del,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                               AS runs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                                             AS wkts
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '2025'
  AND d.innings_number IN (1, 2)
GROUP BY d.match_id, d.innings_number, d.match_date, d.bowling_team, m.team_1, m.team_2
ORDER BY d.match_date, d.innings_number
""")

,match_id,innings_number,match_date,bowling_team,team_1,team_2,deliveries,legal_del,runs,wkts


In [162]:
# Career IPL fielding summary
q(f"""
SELECT
    sum(CASE WHEN wicket_kind = 'caught'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS catches,
    sum(CASE WHEN wicket_kind = 'caught and bowled'
              AND bowler = '{PLAYER}' THEN 1 ELSE 0 END)                         AS caught_and_bowled,
    sum(CASE WHEN wicket_kind = 'stumped'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS stumpings,
    sum(CASE WHEN wicket_kind = 'run out'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
""")

,catches,caught_and_bowled,stumpings,run_outs
0,106.0,3.0,0.0,13.0


In [163]:
# IPL fielding by season
q(f"""
SELECT
    m.season,
    sum(CASE WHEN d.wicket_kind = 'caught'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS catches,
    sum(CASE WHEN d.wicket_kind = 'caught and bowled'
              AND d.bowler = '{PLAYER}' THEN 1 ELSE 0 END)                       AS caught_and_bowled,
    sum(CASE WHEN d.wicket_kind = 'stumped'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS stumpings,
    sum(CASE WHEN d.wicket_kind = 'run out'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
GROUP BY m.season
ORDER BY m.season
""")

,season,catches,caught_and_bowled,stumpings,run_outs
0,2007/08,10.0,0.0,0.0,2.0
1,2009,7.0,0.0,0.0,1.0
2,2009/10,10.0,0.0,0.0,3.0
3,2011,3.0,1.0,0.0,0.0
4,2012,10.0,1.0,0.0,0.0
5,2013,10.0,0.0,0.0,3.0
6,2014,12.0,0.0,0.0,2.0
7,2015,10.0,1.0,0.0,0.0
8,2016,7.0,0.0,0.0,1.0
9,2017,4.0,0.0,0.0,1.0


In [164]:
# ── Diagnostic: all IPL innings for PLAYER ordered by date ───────────────────
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season,
        sum(d.runs_batter) AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END) AS balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, match_date, batting_team, season FROM bat
    UNION
    SELECT match_id, innings_number, match_date, batting_team, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
)
SELECT
    a.season, a.match_date, a.batting_team,
    coalesce(b.runs,  0) AS runs,
    coalesce(b.balls, 0) AS balls,
    (d.match_id IS NOT NULL) AS dismissed
FROM all_inn a
LEFT JOIN bat b USING (match_id, innings_number)
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY a.match_date
""")

,season,match_date,batting_team,runs,balls,dismissed
0,2007/08,2008-04-19,Chennai Super Kings,32.0,13.0,True
1,2007/08,2008-04-23,Chennai Super Kings,53.0,37.0,True
2,2007/08,2008-04-28,Chennai Super Kings,28.0,17.0,True
3,2007/08,2008-05-02,Chennai Super Kings,3.0,4.0,True
4,2007/08,2008-05-04,Chennai Super Kings,27.0,27.0,True
5,2007/08,2008-05-06,Chennai Super Kings,32.0,21.0,True
6,2007/08,2008-05-08,Chennai Super Kings,1.0,2.0,True
7,2007/08,2008-05-10,Chennai Super Kings,26.0,17.0,True
8,2007/08,2008-05-14,Chennai Super Kings,1.0,6.0,True
9,2007/08,2008-05-21,Chennai Super Kings,21.0,20.0,False
